In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame

from src.electricity_price_forecasting.features.full_day_ahead import full_day_ahead_features
from src.electricity_price_forecasting.models.data_split import train_val_test_split

df = full_day_ahead_features(pd.Timestamp("2024-07-01"), pd.Timestamp("2025-04-01"))
train, val, select, test = train_val_test_split(
    df,
    val_cutoff=pd.Timestamp("2025-01-01"),
    select_cutoff=pd.Timestamp("2025-02-01"),
    test_cutoff=pd.Timestamp("2025-03-01"))

/Users/pavelholman/FIT/bp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/pavelholman/FIT/bp/src/electricity_price_forecasting/features/generate_day_ahead_features.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"da_lag_{lag}"] = df["DA Price (EUR/MWh)"].shift(lag)
/Users/pavelholman/FIT/bp/src/electricity_price_forecasting/features/generate_day_ahead_features.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using p

In [2]:
from src.electricity_price_forecasting.models.train_day_ahead import train_day_ahead

predictor, results, leaderboard = train_day_ahead(train, val, select)

2025-04-08 14:32:58,586 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/day_ahead/day_ahead
2025-04-08 14:32:58,586 - INFO - Model not found, training a new model
Beginning AutoGluon training...
AutoGluon will save models to '/Users/pavelholman/FIT/bp/models/day_ahead/day_ahead'
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.12.6
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.3.0: Thu Jan  2 20:24:06 PST 2025; root:xnu-11215.81.4~3/RELEASE_ARM64_T8103
CPU Count:          8
GPU Count:          0
Memory Avail:       3.43 GB / 16.00 GB (21.4%)
Disk Space Avail:   63.54 GB / 460.43 GB (13.8%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAE,
 'freq': 'h',
 'hyperparameters': {'AutoARIMA': {},
                     'AutoETS': {},
                     'Average': {},
                     'Chronos': {},
                     'DirectTabular': {},
               

                                    model  score_test  score_val  pred_time_test  pred_time_val  fit_time_marginal  fit_order
0  Chronos[autogluon__chronos-bolt-small]  -14.928996 -17.639528        1.049181       1.513710           3.444740          8
1                        WeightedEnsemble  -15.405412 -16.597338        1.361044       3.054843           0.424389          9
2                               AutoARIMA  -22.734620 -25.979802        1.144742       2.244970           0.062935          7
3                           SeasonalNaive  -25.826071 -29.527202        0.056221       0.708584           0.003464          2
4                         SeasonalAverage  -26.290357 -18.732411        0.118466       0.216728           0.003466          4
5                                 Average  -29.346905 -24.458095        0.100371       0.090650           0.003329          3
6                                   Naive  -45.320238 -42.927560        0.192214       1.324405           0.003968    

In [46]:
best_model = leaderboard.iloc[0]["model"]
best_model_info = predictor.info()
best_model_params = best_model_info['model_info'][best_model]['hyperparameters']

print(f"Best model: {best_model}")
print(f"Best params: {best_model_params}")
best_model = "Chronos"

Best model: Chronos[autogluon__chronos-bolt-small]
Best params: {}


In [47]:
from autogluon.timeseries.dataset import TimeSeriesDataFrame
from src.electricity_price_forecasting.models.train_day_ahead import train_day_ahead_model

train_val = TimeSeriesDataFrame.from_data_frame(pd.concat([train, val]))
predictor_final = train_day_ahead_model(train_val, select, model_cache="final_model", hyperparameters={best_model: best_model_params})

2025-04-08 15:15:57,464 - INFO - Loading model from /Users/pavelholman/FIT/bp/models/day_ahead/final_model
2025-04-08 15:15:57,464 - INFO - Model not found, training a new model
Beginning AutoGluon training...
AutoGluon will save models to '/Users/pavelholman/FIT/bp/models/day_ahead/final_model'
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.12.6
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.3.0: Thu Jan  2 20:24:06 PST 2025; root:xnu-11215.81.4~3/RELEASE_ARM64_T8103
CPU Count:          8
GPU Count:          0
Memory Avail:       2.98 GB / 16.00 GB (18.7%)
Disk Space Avail:   61.42 GB / 460.43 GB (13.3%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAE,
 'freq': 'h',
 'hyperparameters': {'Chronos': {}},
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 168,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed'

In [56]:
forecast_horizon = predictor_final.prediction_length
freq = predictor_final.freq

start_dates = pd.date_range("2025-03-01", "2025-03-31", freq="D")

results = []

for start_date in start_dates:
    history_df = df[df.index.tz_convert(None) < start_date].copy()
    history_df["timestamp"] = history_df.index.tz_convert(None)
    history_df["item_id"] = "day_ahead"
    ts_input = TimeSeriesDataFrame.from_data_frame(history_df)

    forecast_df = predictor_final.predict(ts_input)

    end_date = start_date + pd.Timedelta(days=1)
    actual_df = df[(df.index.tz_convert(None) >= start_date) & (df.index.tz_convert(None) < end_date)].copy()
    actual_df["timestamp"] = actual_df.index.tz_convert(None)
    actual_df["item_id"] = "day_ahead"

    target = "DA Price (EUR/MWh)"
    merged = forecast_df.merge(actual_df, on=["item_id", "timestamp"], how="inner", suffixes=("_pred", "_true"))
    merged["abs_pct_error"] = (
        abs(merged["mean"] - merged[target]) / merged[target]
    ).replace([float("inf"), float("-inf")], pd.NA).dropna()

    results.append({
        "start_date": start_date,
        "mape": merged["abs_pct_error"].mean(),
        "mae": abs(merged["mean"] - merged[target]).mean(),
        "rmse": ((merged["mean"] - merged[target]) ** 2).mean() ** 0.5,
        "mse": ((merged["mean"] - merged[target]) ** 2).mean(),
        "mae_pct": (abs(merged["mean"] - merged[target]) / merged[target]).mean(),
        "mse_pct": ((merged["mean"] - merged[target]) / merged[target]).mean() ** 2,
        "rmse_pct": ((merged["mean"] - merged[target]) / merged[target]).mean() ** 0.5
    })

results

Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]
Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]
Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]
Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]
Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]
Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]
Model not specified in predict, will default to the model with the best validation score: Chronos[autogluon__chronos-bolt-small]
/var/folders/yb/nq6qgkjd1r1gbjptj_jtzsc80000gn/T/ipykernel_64194/2106256654.py:35: RuntimeWarning

[{'start_date': Timestamp('2025-03-01 00:00:00'),
  'mape': 0.05200329160957718,
  'mae': 6.459969609578451,
  'rmse': 7.69867100857254,
  'mse': 59.269535298235326,
  'mae_pct': 0.05200329160957718,
  'mse_pct': 0.00048579852346072736,
  'rmse_pct': 0.1484615695412491},
 {'start_date': Timestamp('2025-03-02 00:00:00'),
  'mape': 1.009209241339051,
  'mae': 32.35109841664632,
  'rmse': 42.11181071447423,
  'mse': 1773.4046016517066,
  'mae_pct': 1.009209241339051,
  'mse_pct': 1.0124012597690346,
  'rmse_pct': 1.0030860004323696},
 {'start_date': Timestamp('2025-03-03 00:00:00'),
  'mape': 0.3354886854321739,
  'mae': 20.553568102518714,
  'rmse': 26.554011621065367,
  'mse': 705.1155331716745,
  'mae_pct': 0.3354886854321739,
  'mse_pct': 0.06049408393955717,
  'rmse_pct': 0.49593895907455954},
 {'start_date': Timestamp('2025-03-04 00:00:00'),
  'mape': 0.38243436263491987,
  'mae': 21.063071276346843,
  'rmse': 26.424963987195493,
  'mse': 698.2787217245788,
  'mae_pct': 0.3824343626